# Chapter 28: BFS and DFS on Grids and Implicit Graphs

The programs from Chapter 28 of *Data Structures and Algorithms in Python, Hands-On* by Aadhavan Muthurengan, in the book's order. Each code cell is one program; the same program is in this folder as a `.py` file, and its expected output is in the `.out` file with the same name. Benchmark timings differ on every computer and every run.

Run the cells from top to bottom (**Run > Run All Cells**). The programs import each other's files, so keep this notebook in its chapter folder.

## A Grid Is a Graph

`grids.py`

In [1]:
"""Grids as graphs: every cell is a vertex joined to its neighbors up,
down, left and right. No adjacency lists needed: compute the neighbors
on the fly."""

from collections import deque

STEPS = ((1, 0), (-1, 0), (0, 1), (0, -1))


def neighbors(grid, r, c):
    for dr, dc in STEPS:
        nr, nc = r + dr, c + dc
        if 0 <= nr < len(grid) and 0 <= nc < len(grid[0]):
            yield nr, nc


def count_islands(grid):
    """Groups of connected "#" cells. Each new "#" starts a DFS that
    marks its whole island as seen."""
    seen, islands = set(), 0
    for r in range(len(grid)):
        for c in range(len(grid[0])):
            if grid[r][c] == "#" and (r, c) not in seen:
                islands += 1
                stack = [(r, c)]
                seen.add((r, c))
                while stack:
                    cell = stack.pop()
                    for n in neighbors(grid, *cell):
                        if grid[n[0]][n[1]] == "#" and n not in seen:
                            seen.add(n)
                            stack.append(n)
    return islands


def shortest_route(grid, start, goal):
    """Fewest steps from start to goal avoiding "#" walls (BFS), and the
    path; (None, []) if the goal can't be reached."""
    parent = {start: None}
    queue = deque([start])
    while queue:
        cell = queue.popleft()
        if cell == goal:
            path = [cell]
            while parent[path[-1]]:
                path.append(parent[path[-1]])
            return len(path) - 1, path[::-1]
        for n in neighbors(grid, *cell):
            if grid[n[0]][n[1]] != "#" and n not in parent:
                parent[n] = cell
                queue.append(n)
    return None, []


def minutes_to_rot(grid):
    """Every minute, rot spreads from each rotten orange ("R") to fresh
    neighbors ("F"). Minutes until none is fresh, or -1. A BFS that
    starts from all rotten oranges at once (multi-source BFS)."""
    rows = [list(row) for row in grid]
    queue = deque(
        (r, c)
        for r, row in enumerate(rows)
        for c, x in enumerate(row)
        if x == "R"
    )
    fresh = sum(row.count("F") for row in rows)
    minutes = 0
    while queue and fresh:
        minutes += 1
        for _ in range(len(queue)):  # one whole level = one minute
            for nr, nc in neighbors(rows, *queue.popleft()):
                if rows[nr][nc] == "F":
                    rows[nr][nc] = "R"
                    fresh -= 1
                    queue.append((nr, nc))
    return -1 if fresh else minutes


def word_ladder(start, goal, words):
    """Fewest one-letter changes from start to goal, each step a word in
    words. The graph is implicit: neighbors are generated, not stored."""
    words = set(words)
    dist, queue = {start: 0}, deque([start])
    while queue:
        w = queue.popleft()
        if w == goal:
            return dist[w]
        for i in range(len(w)):
            for ch in "abcdefghijklmnopqrstuvwxyz":
                nxt = w[:i] + ch + w[i + 1 :]
                if nxt in words and nxt not in dist:
                    dist[nxt] = dist[w] + 1
                    queue.append(nxt)
    return None

## The Shortest Way Through a Maze

`maze_demo.py`

In [2]:
"""Find the shortest way through a maze and draw it."""

from grids import count_islands, shortest_route, word_ladder

maze = [
    "S..#......",
    ".#.#.####.",
    ".#...#....",
    ".####.#.#.",
    "......#.#G",
]
start, goal = (0, 0), (4, 9)
steps, path = shortest_route(maze, start, goal)
drawn = [list(row) for row in maze]
for r, c in path[1:-1]:
    drawn[r][c] = "*"
print(f"{steps} steps:")
for row in drawn:
    print("".join(row))

islands = ["##..#", "#...#", "..#..", "....#"]
print(count_islands(islands), "islands")
words = ["hot", "dot", "dog", "lot", "log", "cog"]
print(word_ladder("hit", "cog", words), "changes from hit to cog")

17 steps:
S**#******
.#*#*####*
.#***#...*
.####.#.#*
......#.#G
4 islands
4 changes from hit to cog


## Multi-Source BFS

`rot_demo.py`

In [3]:
"""Multi-source BFS: rot spreads from every rotten orange at once."""

from grids import minutes_to_rot

crate = ["RFF.", "FF.F", ".FFF", "...R"]
print(minutes_to_rot(crate), "minutes")
print(minutes_to_rot(["RF", ".F", "F."]), "(the last one is cut off)")

3 minutes
-1 (the last one is cut off)


## Why Not Recursion?

`recursion_limit_dfs.py`

In [4]:
"""Recursive DFS on a big open grid goes thousands of calls deep."""

import sys


def fill_recursive(grid, r, c, seen):
    if not (0 <= r < len(grid) and 0 <= c < len(grid[0])):
        return
    if (r, c) in seen:
        return
    seen.add((r, c))
    for dr, dc in ((1, 0), (-1, 0), (0, 1), (0, -1)):
        fill_recursive(grid, r + dr, c + dc, seen)


print("recursion limit:", sys.getrecursionlimit())
for side in (20, 100):
    grid = ["." * side] * side
    try:
        seen = set()
        fill_recursive(grid, 0, 0, seen)
        print(f"{side} x {side}: filled {len(seen):,} cells")
    except RecursionError:
        print(f"{side} x {side}: RecursionError")

recursion limit: 1000
20 x 20: filled 400 cells
100 x 100: RecursionError


## Timing

`bench_grid_bfs.py`

In [5]:
"""Shortest route across an open square grid of n cells, corner to
corner. BFS visits each cell once: time grows in proportion to n."""

import bench
from grids import shortest_route


def make(n):
    side = int(n**0.5)
    grid = ["." * side] * side
    return grid, (0, 0), (side - 1, side - 1)


sizes = [10_000, 40_000, 160_000, 640_000]
results = {"BFS": bench.doubling(shortest_route, make, sizes)}
bench.table(results)

         n             BFS
    10,000         8.59 ms
    40,000      36 ms x4.2
   160,000     149 ms x4.1
   640,000     732 ms x4.9
